## REASONING MODEL 

In [41]:
import pandas
import numpy

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import math
import heapq

import warnings

import heapq
from dataclasses import dataclass, field
from itertools import count


## PREFERENCE

In [42]:
warnings.filterwarnings("ignore", category = FutureWarning)
torch.set_float32_matmul_precision("high")


## DEVICE HANDLING

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in use: {device}")


Device in use: cuda


## PARAMETER

In [44]:
dummy_input = 200
d_model = 64
d_ff = d_model * 4
num_heads = 4
dropout = 0.1
num_layers = 4
dummy_output = dummy_input
goal_dim = 2
action_dim = 10
state_dim = 256
memory_slots = 32
plan_steps = 5

checkpoint_path = Path("best_reason_model.pt")


## POS ENCODING

In [45]:
def positional_encoding(seq_length, d_model):
    
    pe = torch.zeros(seq_length, d_model)
    pos = torch.arange(0, seq_length, dtype = torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model))
    
    pe[:, 0::2] = torch.sin(pos * div_term)
    pe[:, 1::2] = torch.cos(pos * div_term)
    
    return pe


## HELPER

In [46]:
def safe_tensor(x):
    
    return x if torch.is_tensor(x) else torch.tensor(x, dtype = torch.float32).to(device)


## ENCODER

In [47]:
class encoder_layer(nn.Module):
    
    def __init__(self, d_model = d_model, d_ff = d_ff, num_heads = num_heads, dropout = dropout):
        super(encoder_layer, self).__init__()
        
        # norm
        
        self.norm1 = nn.LayerNorm(d_model)
        
        # attention layer
        
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first = True)
        
        # norm
        
        self.norm2 = nn.LayerNorm(d_model)
        
        # FFN
        
        self.ffn = nn.Sequential(
            
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
        
        # dropout
        
        self.dropout = nn.Dropout(dropout)
        
        # apply optimization
        
        self.apply(self.optim)
        
    def optim(self, modulue):
        
        for m in modulue.modules():
            
            if isinstance(m, nn.Linear):
                
                nn.init.normal_(m.weight, 0, 0.02)
                
                if m.bias is not None:
                    
                    nn.init.zeros_(m.bias)
                    
            if isinstance(m, nn.MultiheadAttention):
                
                nn.init.normal_(m.in_proj_weight, 0, 0.02)
                
                if m.in_proj_bias is not None:
                    
                    nn.init.zeros_(m.in_proj_bias)
                    
                nn.init.normal_(m.out_proj.weight, 0, 0.02)
                
                if m.out_proj.bias is not None:
                    
                    nn.init.zeros_(m.out_proj.bias)
                    
    def forward(self, x, mask = None):
        
        # norm1
        
        norm1 = self.norm1(x)
        
        # mha
        
        attn, _ = self.attn(norm1, norm1, norm1, key_padding_mask = mask)
        
        # residual
        
        x = x + self.dropout(attn)
        
        # norm2
        
        norm2 = self.norm2(x)
        
        # ffn
        
        ffn = self.ffn(norm2)
        
        # residual
        
        x = x + self.dropout(ffn)
        
        return x


In [48]:
class encoder_stack(nn.Module):
    
    def __init__(self, d_model = d_model, num_layers = num_layers):
        super(encoder_stack, self).__init__()
        
        
        # stack
        
        self.layers = nn.ModuleList([
            
            encoder_layer() for layer in range(num_layers)
        ])
        
        # norm
        
        self.norm = nn.LayerNorm(d_model)
        
        
    def forward(self, x, mask = None, Uncover = False):
        
        # now layer stack
        
        for layer in self.layers:
            
            x = layer(x, mask = mask)
            
        if Uncover: print(f'X shape after layer: {x.shape}\nX size: {x.size()}')
        
        # layer norm
        
        norm = self.norm(x)
        
        return norm


## INIT

In [49]:
Encoder = encoder_stack().to(device)
print(Encoder)

print("\n----------------------------------------------")

params = sum(p.numel() for p in Encoder.parameters())
print(f"No of params: {params}")


encoder_stack(
  (layers): ModuleList(
    (0-3): 4 x encoder_layer(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=64, bias=True)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
)

----------------------------------------------
No of params: 200064


## NEURAL MEMORY

In [50]:
class neural_mem(nn.Module):
    
    def __init__(self, d_model = d_model, num_heads = num_heads, state_dim = state_dim, memory_slots = memory_slots):
        super(neural_mem, self).__init__()
        
        self.memory_slots = memory_slots
        self.state_dim = state_dim
        
        # memory
        
        self.memory = nn.Parameter(
            
            torch.randn(memory_slots, state_dim)
        )
        
        # memory attn
        
        self.memory_attn = nn.MultiheadAttention(
            state_dim, num_heads, batch_first = True
        )
        
        # update memory
        
        self.write = nn.Sequential(
            
            nn.Linear(state_dim * 2, state_dim),
            nn.GELU(),
            
            nn.Linear(state_dim, state_dim),
            nn.Sigmoid()
        )
        
        self.update_memory = nn.Sequential(
            
            nn.Linear(state_dim * 2, state_dim),
            nn.GELU(),
            
            nn.Linear(state_dim, state_dim),
            nn.LayerNorm(state_dim)
        )
        
        # state + retrieved memory
        
        self.state_update = nn.Sequential(
            
            nn.Linear(state_dim * 2, state_dim),
            nn.GELU(),
            
            nn.Linear(state_dim, state_dim),
            nn.LayerNorm(state_dim)
        )
        
    def forward(self, state):
        
        batch_size = state.size(0)
        
        # prepare memory
        
        memory = self.memory.unsqueeze(0)
        
        memory = memory.expand(batch_size, -1, -1)
        
        query = state.unsqueeze(1)
        
        retrieved, attn_weight = self.memory_attn(query, memory, memory)
        
        retrieved = retrieved.squeeze(1)
        
        # memory write
        
        write_input = torch.cat([state, retrieved], dim = -1)
        
        write_gate = self.write(write_input)
        
        # memory update
        
        new_memory_content = self.update_memory(write_input)
        
        new_memory = (
            
            (1 - write_gate.unsqueeze(1))
            * memory
            +
            write_gate.unsqueeze(1)
            * new_memory_content.unsqueeze(1)
        )

        # update state
        
        updated_state = self.state_update(write_input)
        
        return (
            updated_state,
            new_memory,
            retrieved,
            attn_weight
        )


## PLANNER

In [51]:
class plan_it(nn.Module):
    
    def __init__(self, state_dim = state_dim, plan_steps = plan_steps, action_dim = action_dim, num_heads = num_heads):
        super(plan_it, self).__init__()
        
        # learnable plan
        
        self.plan = nn.Parameter(
            
            torch.randn(plan_steps, state_dim)
        )
        
        # state projec
        
        self.state_proj = nn.Linear(state_dim, state_dim)
        
        # planner transformation
        
        self.plan_transform = nn.Sequential(
            
            nn.Linear(state_dim, state_dim * 2),
            nn.GELU(),
            
            nn.Linear(state_dim * 2, state_dim),
            nn.LayerNorm(state_dim)
        )
        
        # action for each step
        
        self.action_head = nn.Linear(state_dim, action_dim)
        
        # state conditioned planning
        
        self.plan_attn = nn.MultiheadAttention(state_dim, num_heads, batch_first = True)
        
    def forward(self, state):
        
        batch_size = state.size(0)
        
        state = self.state_proj(state).unsqueeze(1)
        
        queries = self.plan.unsqueeze(0)
        
        queries = queries.expand(batch_size, -1, -1)
        
        # attend plan to state
        
        plan_step, plan_attn = self.plan_attn(queries, state, state)
        
        # plan transformation
        
        plan_states = self.plan_transform(plan_step)
        
        # action logits
        
        action_logits = self.action_head(plan_states)
        
        return plan_states, action_logits, plan_attn
        

## SEARCHER

In [52]:
class search_it(nn.Module):
    
    def __init__(self, state_dim = state_dim, action_dim = action_dim, plan_steps = plan_steps):
        super(search_it, self).__init__()
        
        self.heuristic = nn.Sequential(
            
            nn.Linear(state_dim * 2, state_dim),
            nn.GELU(),
            nn.Linear(state_dim, 1),
            nn.Softplus()
        )
        
        self.step_cost = nn.Sequential(
            
            nn.Linear(state_dim * 2, state_dim),
            nn.GELU(),
            nn.Linear(state_dim, 1),
            nn.Softplus()
        )
        
    def forward(self, model, start_state, goal_rep, max_expansions = 32, actions_per_expansion = 3):
        
        open_set = []
        visited = set()
        tie_breaker = 0
        
        root_h = self.heuristic(torch.cat([start_state, goal_rep], dim = -1)).item()
        heapq.heappush(open_set, (root_h, tie_breaker, start_state, (), 0.0, root_h))
        best_node = None
        
        for _ in range(max_expansions):
            
            if not open_set:
                break
            
            current_f, _, current_state, action_history, current_g, current_h = heapq.heappop(open_set)
            state_key = tuple(torch.round(current_state.squeeze(0) * 100).long().tolist())
            
            if state_key in visited:
                continue
            
            visited.add(state_key)
            
            if best_node is None or current_f < best_node[0]:
                best_node = (current_f, current_state, action_history, current_g, current_h)
            
            action_logits = model.planner(current_state)[1].mean(dim = 1)
            action_ids = action_logits.topk(actions_per_expansion, dim = -1).indices[0]
            
            for action_id in action_ids.tolist():
                
                action = torch.tensor([action_id], dtype = torch.long, device = current_state.device)
                action_rep = model.action_embed(action)
                child_state = model.transition(torch.cat([current_state, action_rep], dim = -1))
                step_cost = self.step_cost(torch.cat([current_state, action_rep], dim = -1)).item()
                child_g = current_g + step_cost
                child_h = self.heuristic(torch.cat([child_state, goal_rep], dim = -1)).item()
                tie_breaker += 1
                heapq.heappush(open_set, (child_g + child_h, tie_breaker, child_state, action_history + (action_id,), child_g, child_h))
        
        return best_node


## NEW SEARCHER

In [53]:
@dataclass(order=True)
class SearchNode:
    
    priority: float
    tie_breaker: int
    state: torch.Tensor = field(compare=False)
    action_history: tuple = field(default_factory=tuple, compare=False)
    path_cost: float = field(default=0.0, compare=False)
    heuristic: float = field(default=0.0, compare=False)

class multi_stage_verifier(nn.Module):
    
    """Goal + state + safety checks; never accepts from goal completion alone."""
    
    def __init__(self, state_dim=state_dim):
        
        super().__init__()
        
        def head():
            
            return nn.Sequential(nn.Linear(state_dim * 2, state_dim), nn.GELU(), nn.Linear(state_dim, 1))
        
        self.goal_check, self.state_check, self.safety_check = head(), head(), head()

    def forward(self, candidate_state, goal_rep):
        
        features = torch.cat([candidate_state, goal_rep], dim=-1)
        goal = torch.sigmoid(self.goal_check(features)).squeeze(-1)
        state = torch.sigmoid(self.state_check(features)).squeeze(-1)
        safety = torch.sigmoid(self.safety_check(features)).squeeze(-1)
        overall = (goal + state + safety) / 3
        decision = torch.where(safety < .5, 2, torch.where((goal >= .5) & (state >= .5), 0, 1))
        
        return {'goal': goal, 'state': state, 'safety': safety, 'overall': overall, 'decision': decision, 'label': ['ACCEPT', 'REPLAN', 'REJECT']}

class best_first_search:
    
    """OPEN-set priority queue, not a beam-limited top-K loop."""
    
    def __init__(self, model, max_expansions=32, actions_per_expansion=3):
        
        self.model, self.max_expansions, self.actions_per_expansion = model, max_expansions, actions_per_expansion

    @torch.no_grad()
    
    def search(self, start_state, goal_rep):
        
        if start_state.size(0) != 1:
            
            raise ValueError('Search plans one task at a time; batch before calling this method.')
        serial, open_set, visited, best = count(), [], set(), None
        start_h = self.model.estimate_heuristic(start_state, goal_rep).item()
        heapq.heappush(open_set, SearchNode(start_h, next(serial), start_state, (), 0.0, start_h))
        
        for _ in range(self.max_expansions):
            
            if not open_set: break
            
            current = heapq.heappop(open_set)
            signature = tuple(torch.round(current.state.squeeze(0) * 100).long().tolist())
            
            if signature in visited: continue
            visited.add(signature)
            verification = self.model.verifier(current.state, goal_rep)
            decision = int(verification['decision'].item())
            
            if decision == 0: return current, verification
            
            if decision != 2 and (best is None or current.priority < best.priority): best = current
            
            action_logits = self.model.planner(current.state)[1].mean(dim=1)
            
            action_ids = action_logits.topk(self.actions_per_expansion, dim=-1).indices[0]
            
            for action_id in action_ids.tolist():
                
                child_state = self.model.transition_state(current.state, action_id)
                path_cost = current.path_cost + self.model.estimate_step_cost(current.state, action_id).item()
                heuristic = self.model.estimate_heuristic(child_state, goal_rep).item()
                heapq.heappush(open_set, SearchNode(path_cost + heuristic, next(serial), child_state, current.action_history + (action_id,), path_cost, heuristic))
                
        best = best or SearchNode(float('inf'), next(serial), start_state, (), 0.0, start_h)
        
        return best, self.model.verifier(best.state, goal_rep)


## MODEL

In [54]:
class reason_model(nn.Module):
    
    def __init__(self, d_model = d_model, state_dim = state_dim, action_dim = action_dim, goal_dim = goal_dim, vocab_input = dummy_input, state_output_dim = 8):
        super(reason_model, self).__init__()

        # Embedding
        
        self.embed = nn.Embedding(vocab_input, d_model)
        
        # Encoder
        
        self.encoder = encoder_stack()
        
        # layer norm
        
        self.norm = nn.LayerNorm(d_model)
        
        # state 
        
        self.state = nn.Linear(d_model, state_dim)
        
        self.state_output = nn.Linear(state_dim, state_output_dim)
        
        # neural memory
        
        self.memory = neural_mem()
        
        # planner
    
        self.planner = plan_it()
        
        # goal
        
        self.goal_rep = nn.Linear(state_dim, state_dim)
        
        self.goal_status = nn.Linear(state_dim, goal_dim)
        
        # search
    
        self.search = search_it()
        
        # verifier
        
        self.verifier = multi_stage_verifier(state_dim=state_dim)
        
        # action embed
        
        self.action_embed = nn.Embedding(action_dim, state_dim)
        
        # transition model
        
        self.transition = nn.Sequential(
            
            nn.Linear(state_dim * 2, state_dim),
            nn.GELU(),
            nn.Linear(state_dim, state_dim),
            nn.LayerNorm(state_dim)
        )
        
        self.transition_output = nn.Linear(state_dim, state_output_dim)
        
        self.search_engine = best_first_search(
            self,
            max_expansions=32,
            actions_per_expansion=3
        )
        
        
        self.d_model = d_model
        
    def forward(
            self,
            x,
            mask=None,
            teacher_action=None,
            run_search=False
        ):

            # ============================================================
            # 1. INPUT → EMBEDDING
            # ============================================================

            seq_length = x.size(1)

            pos_encoded = positional_encoding(
                seq_length,
                self.d_model
            ).to(x.device)

            embed = self.embed(x) * math.sqrt(self.d_model)

            x = embed + pos_encoded


            # ============================================================
            # 2. TRANSFORMER ENCODER
            # ============================================================

            encoded = self.encoder(
                x,
                mask
            )


            # ============================================================
            # 3. POOLING → INITIAL STATE
            # ============================================================

            state_rep = encoded.mean(dim=1)

            norm = self.norm(state_rep)

            state = self.state(norm)


            # ============================================================
            # 4. NEURAL MEMORY
            # ============================================================

            (
                update_state,
                new_memory,
                retrieved_memory,
                attn_weight
            ) = self.memory(state)


            # ============================================================
            # 5. PLANNER
            # ============================================================

            (
                planned_state,
                action_logits,
                plan_attn
            ) = self.planner(update_state)


            # ============================================================
            # 6. GOAL REPRESENTATION
            # ============================================================

            goal_rep = self.goal_rep(update_state)

            goal_status = self.goal_status(update_state)


            # ============================================================
            # 7. SELECT ACTION
            #
            # Training:
            #       teacher_action
            #
            # Normal inference:
            #       planner action
            #
            # Search inference:
            #       Best-First Search action
            # ============================================================

            planner_action = (
                action_logits
                .mean(dim=1)
                .argmax(dim=-1)
            )

            action_id = (
                teacher_action
                if teacher_action is not None
                else planner_action
            )


            # ============================================================
            # 8. BEST-FIRST SEARCH
            # ============================================================

            search_result = None

            if run_search and teacher_action is None:

                search_actions = []
                search_results = []

                with torch.no_grad():

                    for batch_index in range(
                        update_state.size(0)
                    ):

                        start_state = update_state[
                            batch_index:batch_index + 1
                        ]

                        current_goal = goal_rep[
                            batch_index:batch_index + 1
                        ]

                        # --------------------------------------------
                        # Your EXISTING Best-First Search
                        # --------------------------------------------

                        node, verification = (
                            self.search_engine.search(
                                start_state,
                                current_goal
                            )
                        )

                        search_results.append(
                            (node, verification)
                        )

                        # --------------------------------------------
                        # Search returns an action history
                        # --------------------------------------------

                        if (
                            node is not None
                            and len(node.action_history) > 0
                        ):

                            selected_action = (
                                node.action_history[0]
                            )

                        else:

                            # Fallback → planner
                            selected_action = (
                                planner_action[
                                    batch_index
                                ].item()
                            )

                        search_actions.append(
                            selected_action
                        )


                # Convert searched actions back to tensor

                action_id = torch.tensor(
                    search_actions,
                    dtype=torch.long,
                    device=x.device
                )

                # Keep the first search result available
                # for debugging / inspection

                search_result = search_results


            # ============================================================
            # 9. ACTION EMBEDDING
            # ============================================================

            action_rep = self.action_embed(
                action_id
            )


            # ============================================================
            # 10. TRANSITION MODEL
            #
            # Current state + selected action
            #                  ↓
            #              next state
            # ============================================================

            transition_input = torch.cat(
                [
                    update_state,
                    action_rep
                ],
                dim=-1
            )

            next_state = self.transition(
                transition_input
            )


            # ============================================================
            # 11. STATE PREDICTIONS
            # ============================================================

            state_prediction = self.state_output(
                update_state
            )

            next_state_prediction = self.transition_output(
                next_state
            )


            # ============================================================
            # 12. VERIFICATION
            # ============================================================


            verification =  self.verifier(next_state, goal_rep)


            # ============================================================
            # 13. SEARCH EVALUATION
            #
            # f(n) = g(n) + h(n)
            #
            # Here we evaluate the transition actually selected
            # by planner / teacher / Best-First Search.
            # ============================================================

            heuristic = (
                self.search.heuristic(
                    torch.cat(
                        [
                            next_state,
                            goal_rep
                        ],
                        dim=-1
                    )
                )
                .squeeze(-1)
            )

            step_cost = (
                self.search.step_cost(
                    transition_input
                )
                .squeeze(-1)
            )

            course_score = (
                step_cost + heuristic
            ).unsqueeze(-1)


            # ============================================================
            # 14. RETURN
            # ============================================================

            return (
                state,
                action_logits,
                action_id,
                goal_rep,
                goal_status,
                course_score,
                planned_state,
                plan_attn,
                next_state,
                update_state,
                new_memory,
                retrieved_memory,
                attn_weight,
                state_prediction,
                next_state_prediction,
                verification,
                heuristic,
                step_cost
            )


## DATA PREPROCESSING & VOCABULARY

In [55]:
import os
import json
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# Token to ID vocabulary builder for task text prompts
VOCAB_MAP = {"<PAD>": 0, "<UNK>": 1}
DATASET_DIR = Path("c:/Dominance/reasoning_dataset_v2")

def build_vocabulary(dataset_path=DATASET_DIR / "train.json"):
    global VOCAB_MAP
    if not dataset_path.exists():
        dataset_path = DATASET_DIR / "all.json"
        if not dataset_path.exists():
            print(f"Warning: Dataset file not found. Using default dummy vocab.")
            return VOCAB_MAP
    
    print(f"Building vocabulary from {dataset_path}...")
    with open(dataset_path, "r", encoding="utf-8") as f:
        records = json.load(f)
    
    for record in records:
        text = record["task"].replace(".", "").replace(",", "").lower()
        for token in text.split():
            if token not in VOCAB_MAP:
                VOCAB_MAP[token] = len(VOCAB_MAP)
                
    return VOCAB_MAP

build_vocabulary()
print(f"Vocabulary built successfully. Total unique tokens: {len(VOCAB_MAP)}")

def text_to_tensor(text, max_length=32):
    tokens = text.replace(".", "").replace(",", "").lower().split()
    ids = [VOCAB_MAP.get(token, VOCAB_MAP["<UNK>"]) for token in tokens[:max_length]]
    if len(ids) < max_length:
        ids += [VOCAB_MAP["<PAD>"]] * (max_length - len(ids))
    return torch.tensor(ids, dtype=torch.long)


Building vocabulary from c:\Dominance\reasoning_dataset_v2\train.json...
Vocabulary built successfully. Total unique tokens: 44


## PYTORCH DATASET & DATALOADER

In [56]:
DECISION_LABEL_MAP = {"ACCEPT": 0, "REPLAN": 1, "REJECT": 2}

class ReasoningTrajectoryDataset(Dataset):
    def __init__(self, json_file, max_seq_len=32, max_trajectories=20000):
        self.json_file = Path(json_file)
        print(f"Loading {self.json_file.name}...")
        with open(self.json_file, "r", encoding="utf-8") as f:
            trajectories = json.load(f)
        
        # Sub-sample trajectories if max_trajectories specified to optimize memory & speed
        if max_trajectories and len(trajectories) > max_trajectories:
            print(f"Sub-sampling dataset from {len(trajectories):,} to {max_trajectories:,} trajectories for optimal training speed.")
            trajectories = trajectories[:max_trajectories]
            
        self.samples = []
        for traj in trajectories:
            task_tensor = text_to_tensor(traj["task"], max_length=max_seq_len)
            for step in traj["steps"]:
                sample = {
                    "task": task_tensor,
                    "curr_state": torch.tensor(step["state"]["vector"], dtype=torch.float32),
                    "next_state": torch.tensor(step["next_state"]["vector"], dtype=torch.float32),
                    "action_id": torch.tensor(step["selected_action_id"], dtype=torch.long),
                    "action_valid": torch.tensor(float(step["action_valid"]), dtype=torch.float32),
                    "step_cost": torch.tensor(float(step["step_cost"]), dtype=torch.float32),
                    "remaining_cost": torch.tensor(float(step["remaining_cost"]), dtype=torch.float32),
                    "goal_reached": torch.tensor(float(step["goal_reached"]), dtype=torch.float32),
                    "state_valid": torch.tensor(float(step["state_valid"]), dtype=torch.float32),
                    "safe": torch.tensor(float(step["safe"]), dtype=torch.float32),
                    "verification_decision": torch.tensor(DECISION_LABEL_MAP.get(step["verification_decision"], 1), dtype=torch.long)
                }
                self.samples.append(sample)
                
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        return self.samples[idx]

# Create Datasets and DataLoaders (Sub-sampled train set for peak throughput + solid convergence)
batch_size = 128

train_dataset = ReasoningTrajectoryDataset(DATASET_DIR / "train.json", max_trajectories=20000)
val_dataset = ReasoningTrajectoryDataset(DATASET_DIR / "validation.json", max_trajectories=3000)
test_dataset = ReasoningTrajectoryDataset(DATASET_DIR / "test.json", max_trajectories=3000)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"DataLoaders Initialized:")
print(f"  Train samples: {len(train_dataset):,} ({len(train_loader):,} batches)")
print(f"  Val samples:   {len(val_dataset):,} ({len(val_loader):,} batches)")
print(f"  Test samples:  {len(test_dataset):,} ({len(test_loader):,} batches)")


Loading train.json...
Sub-sampling dataset from 80,000 to 20,000 trajectories for optimal training speed.
Loading validation.json...
Sub-sampling dataset from 10,000 to 3,000 trajectories for optimal training speed.
Loading test.json...
Sub-sampling dataset from 10,000 to 3,000 trajectories for optimal training speed.
DataLoaders Initialized:
  Train samples: 217,867 (1,703 batches)
  Val samples:   32,705 (256 batches)
  Test samples:  32,487 (254 batches)


## MULTI-TASK LOSS FUNCTION

In [57]:
class ReasoningMultiTaskLoss(nn.Module):
    """Multi-Task Loss function combining all supervised signals with numerical stability."""
    def __init__(self, w_action=1.0, w_state=1.0, w_transition=1.0, w_cost=0.5, w_heuristic=0.5, w_verify=1.0):
        super().__init__()
        self.w_action = w_action
        self.w_state = w_state
        self.w_transition = w_transition
        self.w_cost = w_cost
        self.w_heuristic = w_heuristic
        self.w_verify = w_verify
        
        self.ce_loss = nn.CrossEntropyLoss()
        self.bce_logits_loss = nn.BCEWithLogitsLoss()
        self.bce_loss = nn.BCELoss()  # for verifier outputs which already pass through Sigmoid
        self.mse_loss = nn.MSELoss()
        
    def forward(self, outputs, batch):
        (
            state, action_logits, action_id, goal_rep, goal_status,
            course_score, planned_state, plan_attn, next_state, update_state,
            new_memory, retrieved_memory, attn_weight, state_prediction, next_state_prediction,
            verification, heuristic, step_cost
        ) = outputs
        
        # 1. Action selection loss (Planner actions over plan steps)
        target_action = batch["action_id"].to(state.device)
        logits_flat = action_logits.view(-1, action_logits.size(-1))
        target_action_flat = target_action.unsqueeze(1).repeat(1, action_logits.size(1)).view(-1)
        loss_action = self.ce_loss(logits_flat, target_action_flat)
        
        # 2. Current state vector prediction loss (BCEWithLogitsLoss for numerical stability)
        target_curr_state = batch["curr_state"][:, :state_prediction.size(-1)].to(state.device)
        loss_state = self.bce_logits_loss(state_prediction, target_curr_state)
        
        # 3. Next state / transition prediction loss (BCEWithLogitsLoss)
        target_next_state = batch["next_state"][:, :next_state_prediction.size(-1)].to(state.device)
        loss_transition = self.bce_logits_loss(next_state_prediction, target_next_state)
        
        # 4. Step cost & heuristic prediction losses
        target_step_cost = batch["step_cost"].to(state.device)
        target_remaining_cost = batch["remaining_cost"].to(state.device)
        loss_cost = self.mse_loss(step_cost, target_step_cost)
        loss_heuristic = self.mse_loss(heuristic, target_remaining_cost)
        
        # 5. Multi-stage verifier losses
        goal_prob = verification["goal"]
        state_prob = verification["state"]
        safety_prob = verification["safety"]

        loss_goal_verify = self.bce_loss(goal_prob, batch["goal_reached"].to(state.device))
        loss_state_verify = self.bce_loss(state_prob, batch["state_valid"].to(state.device))
        loss_safety_verify = self.bce_loss(safety_prob, batch["safe"].to(state.device))
        loss_verify = (loss_goal_verify + loss_state_verify + loss_safety_verify) / 3.0
        
        total_loss = (
            self.w_action * loss_action +
            self.w_state * loss_state +
            self.w_transition * loss_transition +
            self.w_cost * loss_cost +
            self.w_heuristic * loss_heuristic +
            self.w_verify * loss_verify
        )
        
        return total_loss, {
            "loss_action": loss_action.item(),
            "loss_state": loss_state.item(),
            "loss_transition": loss_transition.item(),
            "loss_cost": loss_cost.item(),
            "loss_heuristic": loss_heuristic.item(),
            "loss_verify": loss_verify.item()
        }

criterion = ReasoningMultiTaskLoss().to(device)


## TRAINING & VALIDATION LOOP

In [58]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    metrics_sum = {}
    correct_actions = 0
    total_samples = 0
    
    for batch in dataloader:
        x = batch["task"].to(device)
        teacher_action = batch["action_id"].to(device)
        
        optimizer.zero_grad()
        outputs = model(x, teacher_action=teacher_action)
        loss, sub_losses = criterion(outputs, batch)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        b_size = x.size(0)
        running_loss += loss.item() * b_size
        total_samples += b_size
        
        for k, v in sub_losses.items():
            metrics_sum[k] = metrics_sum.get(k, 0.0) + v * b_size
            
        action_logits = outputs[1]
        pred_action = action_logits.mean(dim=1).argmax(dim=-1)
        correct_actions += (pred_action == teacher_action).sum().item()
        
    epoch_loss = running_loss / total_samples
    action_acc = correct_actions / total_samples
    avg_sub_losses = {k: v / total_samples for k, v in metrics_sum.items()}
    
    return epoch_loss, action_acc, avg_sub_losses

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    metrics_sum = {}
    correct_actions = 0
    total_samples = 0
    
    for batch in dataloader:
        x = batch["task"].to(device)
        teacher_action = batch["action_id"].to(device)
        
        outputs = model(x, teacher_action=teacher_action)
        loss, sub_losses = criterion(outputs, batch)
        
        b_size = x.size(0)
        running_loss += loss.item() * b_size
        total_samples += b_size
        
        for k, v in sub_losses.items():
            metrics_sum[k] = metrics_sum.get(k, 0.0) + v * b_size
            
        action_logits = outputs[1]
        pred_action = action_logits.mean(dim=1).argmax(dim=-1)
        correct_actions += (pred_action == teacher_action).sum().item()
        
    val_loss = running_loss / total_samples
    action_acc = correct_actions / total_samples
    avg_sub_losses = {k: v / total_samples for k, v in metrics_sum.items()}
    
    return val_loss, action_acc, avg_sub_losses


## MODEL TRAINING EXECUTION

In [59]:
vocab_size = len(VOCAB_MAP)
model = reason_model(vocab_input=vocab_size).to(device)


In [60]:
# Initialize Model and Optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

num_epochs = 20
best_val_loss = float("inf")

print(f"Starting Training for {num_epochs} Epochs on device: {device}")
print("=" * 75)

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc, train_sub = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, val_sub = evaluate(model, val_loader, criterion, device)
    
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)
        saved_str = "[SAVED]"
    else:
        saved_str = ""
        
    print(f"Epoch {epoch:02d}/{num_epochs:02d} | Train Loss: {train_loss:.4f} (Acc: {train_acc*100:.2f}%) | Val Loss: {val_loss:.4f} (Acc: {val_acc*100:.2f}%) {saved_str}")
    print(f"   L_act: {val_sub['loss_action']:.3f} | L_state: {val_sub['loss_state']:.3f} | L_trans: {val_sub['loss_transition']:.3f} | L_cost: {val_sub['loss_cost']:.3f} | L_ver: {val_sub['loss_verify']:.3f}")


Starting Training for 20 Epochs on device: cuda
Epoch 01/20 | Train Loss: 3.4764 (Acc: 21.78%) | Val Loss: 3.4021 (Acc: 21.38%) [SAVED]
   L_act: 1.981 | L_state: 0.329 | L_trans: 0.156 | L_cost: 0.178 | L_ver: 0.115
Epoch 02/20 | Train Loss: 3.4022 (Acc: 22.13%) | Val Loss: 3.4152 (Acc: 22.37%) 
   L_act: 1.980 | L_state: 0.329 | L_trans: 0.161 | L_cost: 0.172 | L_ver: 0.115
Epoch 03/20 | Train Loss: 3.3974 (Acc: 22.32%) | Val Loss: 3.4142 (Acc: 22.61%) 
   L_act: 1.979 | L_state: 0.329 | L_trans: 0.157 | L_cost: 0.175 | L_ver: 0.115
Epoch 04/20 | Train Loss: 3.3930 (Acc: 22.31%) | Val Loss: 3.4144 (Acc: 22.37%) 
   L_act: 1.978 | L_state: 0.329 | L_trans: 0.159 | L_cost: 0.182 | L_ver: 0.115
Epoch 05/20 | Train Loss: 3.3842 (Acc: 22.50%) | Val Loss: 3.3967 (Acc: 22.61%) [SAVED]
   L_act: 1.976 | L_state: 0.328 | L_trans: 0.155 | L_cost: 0.170 | L_ver: 0.115
Epoch 06/20 | Train Loss: 3.3841 (Acc: 22.48%) | Val Loss: 3.3839 (Acc: 22.37%) [SAVED]
   L_act: 1.976 | L_state: 0.328 | L_tra

## TEST LOOP & EVALUATION

In [61]:
# Load best checkpoint and evaluate on Test set
if checkpoint_path.exists():
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Loaded best model parameters from {checkpoint_path}")

test_loss, test_acc, test_sub = evaluate(model, test_loader, criterion, device)

print("\n" + "=" * 75)
print("FINAL TEST EVALUATION RESULTS")
print("=" * 75)
print(f"Test Loss:             {test_loss:.4f}")
print(f"Action Prediction Acc: {test_acc*100:.2f}%")
print(f"Action Loss:           {test_sub['loss_action']:.4f}")
print(f"State Predict Loss:    {test_sub['loss_state']:.4f}")
print(f"Transition Loss:       {test_sub['loss_transition']:.4f}")
print(f"Step Cost Loss:        {test_sub['loss_cost']:.4f}")
print(f"Heuristic Loss:        {test_sub['loss_heuristic']:.4f}")
print(f"Verifier Loss:         {test_sub['loss_verify']:.4f}")
print("=" * 75)


Loaded best model parameters from best_reason_model.pt

FINAL TEST EVALUATION RESULTS
Test Loss:             3.3201
Action Prediction Acc: 22.61%
Action Loss:           1.9632
State Predict Loss:    0.3236
Transition Loss:       0.1494
Step Cost Loss:        0.1606
Heuristic Loss:        1.3853
Verifier Loss:         0.1110


In [62]:
state_dict = torch.load(
    r"C:\Dominance\Self-Perpetuating-Model\Reasoning model\best_reason_model.pt",
    map_location=device,
    weights_only=True
)

model = reason_model(vocab_input = vocab_size)
model.load_state_dict(state_dict)
model.to(device)


reason_model(
  (embed): Embedding(44, 64)
  (encoder): encoder_stack(
    (layers): ModuleList(
      (0-3): 4 x encoder_layer(
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (ffn): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=64, bias=True)
        )
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (state): Linear(in_features=64, out_features=256, bias=True)
  (state_output): Linear(in_features=256, out_features=8, bias=True)
  (memory): neural_mem(
    (memory_attn): Multih

In [64]:
ACTION_MAP = {
    0: "inspect", 1: "prepare", 2: "build", 3: "execute", 4: "test",
    5: "repair", 6: "observe", 7: "verify", 8: "export", 9: "no_op"
}

def predict_reasoning(task_prompt, model, target_device=device):
    #model = model.to(target_device)
    model.eval()
    
    # 1. Tokenize prompt text
    x = text_to_tensor(task_prompt, max_length=32).unsqueeze(0).to(target_device)
    
    with torch.no_grad():
        outputs = model(x, run_search=False)
        
    (
        state, action_logits, action_id, goal_rep, goal_status,
        course_score, planned_state, plan_attn, next_state, update_state,
        new_memory, retrieved_memory, attn_weight, state_prediction, next_state_prediction,
        verification, heuristic, step_cost
    ) = outputs
    
    # 2. Decode raw tensors into human-interpretable values
    pred_action_id = action_id[0].item() if action_id.dim() > 0 else action_id.item()
    action_name = ACTION_MAP.get(pred_action_id, f"action_{pred_action_id}")
    
    goal_prob, state_prob, safety_prob = verification['goal'].item(), verification['state'].item(), verification['safety'].item()
    overall_score = verification['overall'].item()
    decision_idx = int(verification['decision'].item()) if torch.is_tensor(verification['decision']) else int(verification['decision'])
    verdict = verification['label'][decision_idx]
        
    top_plan_actions = action_logits.mean(dim=1).topk(3, dim=-1).indices[0].tolist()
    top_plan_names = [ACTION_MAP.get(idx, f"action_{idx}") for idx in top_plan_actions]

    # 3. Print clean readable interface output
    print("=" * 65)
    print(f"INPUT TASK:           {task_prompt}")
    print("=" * 65)
    print(f"VERIFIER DECISION:    {verdict} (Overall Confidence: {overall_score*100:.1f}%)")
    print(f"  [+] Goal Completion: {goal_prob*100:.1f}%")
    print(f"  [+] State Validity:  {state_prob*100:.1f}%")
    print(f"  [+] Safety Check:    {safety_prob*100:.1f}%")
    print("-" * 65)
    print(f"RECOMMENDED ACTION:   {action_name.upper()} (ID: {pred_action_id})")
    print(f"PLANNED SEQUENCE:     {' -> '.join(top_plan_names)}")
    print(f"PREDICTED STEP COST:  {step_cost.item():.4f}")
    print(f"ESTIMATED REMAINING:  {heuristic.item():.4f}")
    print("=" * 65 + "\n")

# Run Inference Example
predict_reasoning("Clean a customer dataset and export a validated file.", model)


INPUT TASK:           Clean a customer dataset and export a validated file.
VERIFIER DECISION:    REPLAN (Overall Confidence: 63.8%)
  [+] Goal Completion: 0.0%
  [+] State Validity:  91.4%
  [+] Safety Check:    100.0%
-----------------------------------------------------------------
RECOMMENDED ACTION:   INSPECT (ID: 0)
PLANNED SEQUENCE:     inspect -> execute -> prepare
PREDICTED STEP COST:  1.0834
ESTIMATED REMAINING:  6.9426

